# What If Polynomials Aren't Enough?

In Notebook 1, you fit a polynomial to your robot's data by minimizing error. You figured out how to choose the right degree without cheating (cross-validation). That process — fitting a function to data by minimizing error — **is** machine learning.

Polynomial regression is the simplest ML model.

But what if the relationship between input and output is too complex for any polynomial?

---

## Part 1: A Neural Net You Can Read

Before using any library, let's build a neural net from code you can read line by line.

This is [micrograd](https://github.com/karpathy/micrograd) by Andrej Karpathy (ex-OpenAI, ex-Tesla AI). The entire neural net engine is two files, ~120 lines of Python, no dependencies. Every operation records how to compute its gradient — that's the chain rule from your calc class, applied automatically.

### The engine: automatic differentiation

Read this code. Every `_backward` function is the chain rule for one operation:
- Addition: `d/dx (x + y) = 1`, so both inputs get `out.grad`
- Multiplication: `d/dx (x * y) = y`, so `self.grad += other.data * out.grad`
- Power: `d/dx (x^n) = n * x^(n-1)`
- ReLU: gradient passes through if input > 0, otherwise 0

This is Calc BC. You've computed all of these derivatives by hand.

In [ ]:
# micrograd engine.py — Andrej Karpathy
# https://github.com/karpathy/micrograd/blob/master/micrograd/engine.py

class Value:
    """Stores a single scalar value and its gradient."""

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad          # d/dx (x + y) = 1
            other.grad += out.grad         # d/dy (x + y) = 1
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad   # d/dx (x * y) = y
            other.grad += self.data * out.grad   # d/dy (x * y) = x
        out._backward = _backward
        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data ** (other - 1)) * out.grad  # power rule
        out._backward = _backward
        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')
        def _backward():
            self.grad += (out.data > 0) * out.grad  # pass gradient through if > 0
        out._backward = _backward
        return out

    def backward(self):
        # Topological sort — process nodes in the right order
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # Apply the chain rule backward through the graph
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __neg__(self):      return self * -1
    def __radd__(self, other): return self + other
    def __sub__(self, other):  return self + (-other)
    def __rsub__(self, other): return other + (-self)
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def __rtruediv__(self, other): return other * self**-1
    def __repr__(self): return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

### The neural net: layers of neurons

A neuron computes `y = ReLU(w₁·x₁ + w₂·x₂ + ... + b)`. A layer is a list of neurons. An MLP is a list of layers. That's the entire architecture.

In [ ]:
# micrograd nn.py — Andrej Karpathy
# https://github.com/karpathy/micrograd/blob/master/micrograd/nn.py

import random

class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0
    def parameters(self):
        return []

class Neuron(Module):
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.relu() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

class Layer(Module):
    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin, **kwargs) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP(Module):
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1], nonlin=i != len(nouts)-1)
                       for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

### Train it on your robot's data

Same 13 data points from Notebook 1. Same goal: predict power from distance. But now with a neural net instead of a polynomial.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Your data
distances = np.array([47, 50, 57, 60, 65, 68, 72, 76, 83, 90, 95, 100, 110], dtype=np.float64)
power = np.array([975, 985, 1000, 1005, 1010, 1012, 1020, 1025, 1031, 1080, 1120, 1150, 1200], dtype=np.float64)

# Normalize (neural nets learn better when inputs/outputs are roughly -1 to 1)
d_mean, d_std = distances.mean(), distances.std()
p_mean, p_std = power.mean(), power.std()
d_norm = (distances - d_mean) / d_std
p_norm = (power - p_mean) / p_std

# Build a tiny net: 1 input -> 10 neurons -> 10 neurons -> 1 output
net = MLP(1, [10, 10, 1])
print(f"Number of parameters: {len(net.parameters())}")

In [ ]:
# Training loop — this is the whole algorithm:
# 1. Forward pass: compute predictions
# 2. Compute loss (how wrong are we?)
# 3. Backward pass: chain rule through the whole graph
# 4. Update weights: nudge each parameter to reduce loss

learning_rate = 0.01
losses = []

for epoch in range(200):
    # Forward: predict each point
    preds = [net([Value(d)]) for d in d_norm]
    
    # Loss: mean squared error
    loss = sum((p - Value(t)) ** 2 for p, t in zip(preds, p_norm)) * (1.0 / len(preds))
    
    # Backward: compute gradients (the chain rule!)
    net.zero_grad()
    loss.backward()
    
    # Update: gradient descent step
    for p in net.parameters():
        p.data -= learning_rate * p.grad
    
    losses.append(loss.data)
    if epoch % 50 == 0:
        print(f"Epoch {epoch}: loss = {loss.data:.6f}")

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss — is it learning?')
plt.show()

### Compare: neural net vs. your polynomial

Plot the neural net's predictions alongside your degree-2 polynomial from Notebook 1.

For 13 data points and a smooth quadratic relationship — which would you trust more? Why?

In [ ]:
# Generate smooth predictions from the neural net
x_smooth = np.linspace(distances.min(), distances.max(), 200)
x_smooth_norm = (x_smooth - d_mean) / d_std
y_smooth_norm = [net([Value(d)]).data for d in x_smooth_norm]
y_smooth = np.array(y_smooth_norm) * p_std + p_mean

# Your degree-2 polynomial from Notebook 1
coeffs = np.polyfit(distances, power, 2)
y_poly = np.polyval(coeffs, x_smooth)

plt.scatter(distances, power, color='black', label='Measured data', zorder=5)
plt.plot(x_smooth, y_smooth, label='Neural net (micrograd)', color='blue')
plt.plot(x_smooth, y_poly, label='Degree-2 polynomial', color='red', linestyle='--')
plt.xlabel('Distance')
plt.ylabel('Power')
plt.legend()
plt.title('Neural net vs. polynomial — which do you trust?')
plt.show()

The polynomial probably wins here. For 13 data points with a smooth quadratic relationship, a degree-2 polynomial is the right tool — it matches the physics, it's simple, and it generalizes.

The neural net has ~130 parameters trying to fit 13 points. It might memorize them (overfitting!) or produce a wiggly curve. More parameters than data points is a recipe for trouble.

**So why bother with neural nets?** Because polynomials can't do everything. They can't classify images. They can't handle 784 input dimensions. When the real pattern is complex and you have lots of data, neural nets shine. Polynomials break.

### Look inside

Look at the actual numbers the network learned. These are the weights and biases — just numbers that got adjusted, step by step, to reduce the error.

In [ ]:
for i, layer in enumerate(net.layers):
    print(f"\nLayer {i}: {len(layer.neurons)} neurons")
    for j, neuron in enumerate(layer.neurons):
        weights = [w.data for w in neuron.w]
        print(f"  Neuron {j}: weights={[f'{w:.3f}' for w in weights]}, bias={neuron.b.data:.3f}")

---

## Part 2: Can Your Net See?

Your robot's data had 1 input (distance). What about images? An 8×8 image has 64 pixels — 64 inputs. Same micrograd code, same chain rule, just more numbers going in.

We'll use real handwritten digits — small (8×8) so micrograd can handle them interactively. You can inspect every forward pass, every gradient, at the level of individual Values.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
# digits.images: (1797, 8, 8) — real handwritten digits
# digits.target: (1797,) — labels 0-9
# Pixel values: 0 (white) to 16 (black)

# Look at some
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray_r')
    ax.set_title(f'Label: {digits.target[i]}')
    ax.axis('off')
plt.suptitle('Handwritten digits — 8×8 pixels each')
plt.tight_layout()
plt.show()

In [ ]:
# What does one image look like as numbers?
print("A digit 0 — the raw pixel values:")
print(digits.images[0])
print(f"\nShape: {digits.images[0].shape}")
print(f"Min pixel: {digits.images[0].min()}, Max pixel: {digits.images[0].max()}")
print(f"\nFlattened to {digits.images[0].size} numbers — this is what the network sees.")

### Start simple: just 0 vs 1

Before trying all 10 digits, start with just two: 0 and 1. They look very different — the network should be able to tell them apart.

One output neuron: close to 0 → it's a zero, close to 1 → it's a one. Same MSE loss you used in Notebook 1.

In [ ]:
# Filter to just 0s and 1s
mask = (digits.target == 0) | (digits.target == 1)
X_bin = digits.data[mask] / 16.0   # normalize pixels to 0-1
y_bin = digits.target[mask]         # 0 or 1

print(f"{len(X_bin)} images: {(y_bin==0).sum()} zeros, {(y_bin==1).sum()} ones")
print(f"Each image: {X_bin.shape[1]} pixels (8×8 flattened)")

# Shuffle and split: 80% train, 20% test
random.seed(42)
np.random.seed(42)
perm = np.random.permutation(len(X_bin))
split = int(0.8 * len(X_bin))
X_train_bin, y_train_bin = X_bin[perm[:split]], y_bin[perm[:split]]
X_test_bin, y_test_bin = X_bin[perm[split:]], y_bin[perm[split:]]

In [ ]:
# Build network: 64 pixels -> 16 hidden neurons -> 1 output
random.seed(42)
net2 = MLP(64, [16, 1])
print(f"Parameters: {len(net2.parameters())}")

# Train — one image at a time (SGD)
# Micrograd processes one number at a time, so this keeps the computation graph small and fast.
learning_rate = 0.05
train_losses = []

for epoch in range(10):
    indices = list(range(len(X_train_bin)))
    random.shuffle(indices)
    
    total_loss = 0
    correct = 0
    
    for i in indices:
        # Wrap each pixel in a Value — so gradients flow back to the input
        x = [Value(float(p)) for p in X_train_bin[i]]
        pred = net2(x)
        target = float(y_train_bin[i])
        loss = (pred - target) ** 2
        
        net2.zero_grad()
        loss.backward()
        
        for p in net2.parameters():
            p.data -= learning_rate * p.grad
        
        total_loss += loss.data
        correct += ((pred.data > 0.5) == y_train_bin[i])
    
    avg_loss = total_loss / len(X_train_bin)
    train_acc = correct / len(X_train_bin)
    
    # Test accuracy
    test_correct = 0
    for i in range(len(X_test_bin)):
        x = [Value(float(p)) for p in X_test_bin[i]]
        pred = net2(x)
        test_correct += ((pred.data > 0.5) == y_test_bin[i])
    test_acc = test_correct / len(X_test_bin)
    
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1:2d}: loss = {avg_loss:.4f}, train acc = {train_acc:.1%}, test acc = {test_acc:.1%}")

### Trace a prediction: the forward pass

Pick one image. Feed it through the network layer by layer. Watch the 64 pixel values get transformed into 16 hidden activations, then into 1 prediction.

Every number here is a `Value` object — the same class you read above. Every multiply and add is being recorded in the computation graph.

In [ ]:
# Pick a test image
sample_idx = 0
x = [Value(float(p)) for p in X_test_bin[sample_idx]]

# Forward pass — layer by layer, so we can inspect intermediate values
hidden = net2.layers[0](x)     # 64 pixels -> 16 hidden neurons (with ReLU)
output = net2.layers[1](hidden) # 16 hidden -> 1 output (no ReLU)

hidden_acts = [h.data for h in hidden]
prediction = output.data
true_label = y_test_bin[sample_idx]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

# 1. Input image
axes[0].imshow(X_test_bin[sample_idx].reshape(8, 8), cmap='gray_r')
axes[0].set_title(f'Input (true label: {true_label})')
axes[0].axis('off')

# 2. Hidden layer activations — what the network "computed" from the pixels
colors = ['steelblue' if a > 0 else 'lightcoral' for a in hidden_acts]
axes[1].bar(range(16), hidden_acts, color=colors)
axes[1].set_title('Hidden layer: 16 neuron activations')
axes[1].set_xlabel('Neuron')
axes[1].set_ylabel('Activation (after ReLU)')

# 3. Output
axes[2].bar([0], [prediction], color='steelblue')
axes[2].axhline(0.5, color='red', linestyle='--', label='Decision boundary')
axes[2].set_title(f'Output: {prediction:.3f} → predicts {"1" if prediction > 0.5 else "0"}')
axes[2].set_ylabel('Output value')
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"\n64 pixels → 16 hidden values → 1 prediction.")
print(f"Every arrow in this chain is a multiply-and-add, tracked by a Value object.")

### Trace a prediction: the backward pass

Now the chain rule. Call `loss.backward()` and every `Value` in the graph gets its gradient set. Including the **input pixels** — their gradients tell you which pixels the network thinks matter most for this prediction.

This is called a **saliency map**. It's a real technique used in ML research — same math, same chain rule.

In [ ]:
# Backward pass — compute gradients all the way back to the input pixels
x = [Value(float(p)) for p in X_test_bin[sample_idx]]  # fresh Values
pred = net2(x)
target = float(y_test_bin[sample_idx])
loss = (pred - target) ** 2

net2.zero_grad()
loss.backward()

# The gradient on each input pixel: how much would changing this pixel reduce the loss?
input_grads = np.array([v.grad for v in x])
saliency = np.abs(input_grads).reshape(8, 8)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

axes[0].imshow(X_test_bin[sample_idx].reshape(8, 8), cmap='gray_r')
axes[0].set_title(f'Input (label: {y_test_bin[sample_idx]})')
axes[0].axis('off')

axes[1].imshow(saliency, cmap='hot')
axes[1].set_title('Saliency: which pixels matter?')
axes[1].axis('off')

axes[2].imshow(input_grads.reshape(8, 8), cmap='RdBu', vmin=-np.abs(input_grads).max(), vmax=np.abs(input_grads).max())
axes[2].set_title('Signed gradients (red = increase, blue = decrease)')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"The chain rule flowed backward from the loss, through the output layer,")
print(f"through the hidden layer, all the way to each of the 64 input pixels.")
print(f"That's backpropagation. Same code you read in engine.py.")

### What is each neuron looking for?

Each of the 16 hidden neurons has 64 weights — one per pixel. Reshape those weights to 8×8 and you can *see* the pattern that neuron responds to.

These aren't the sharp "edge detectors" you might see in articles about deep learning (those come from **convolutional** networks with many more layers). With 16 neurons and 8×8 images, the patterns are simpler — more like blurry templates. But the same principle applies: each neuron specializes.

In [ ]:
# Visualize first-layer weights: what pattern does each neuron respond to?
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    weights = np.array([w.data for w in net2.layers[0].neurons[i].w])
    ax.imshow(weights.reshape(8, 8), cmap='RdBu', vmin=-np.abs(weights).max(), vmax=np.abs(weights).max())
    ax.set_title(f'N{i}', fontsize=9)
    ax.axis('off')
plt.suptitle('First-layer weights — what each neuron is looking for', fontsize=13)
plt.tight_layout()
plt.show()

### Which neurons respond to which digit?

Feed several 0s and several 1s through the network. Compare the hidden layer activations. Do any neurons specialize — firing for one digit but not the other?

In [ ]:
# Compare hidden activations for 0s vs 1s
acts_0, acts_1 = [], []
for i in range(len(X_test_bin)):
    x = [Value(float(p)) for p in X_test_bin[i]]
    hidden = net2.layers[0](x)
    acts = [h.data for h in hidden]
    if y_test_bin[i] == 0:
        acts_0.append(acts)
    else:
        acts_1.append(acts)

mean_acts_0 = np.array(acts_0).mean(axis=0)
mean_acts_1 = np.array(acts_1).mean(axis=0)

fig, ax = plt.subplots(figsize=(10, 4))
x_pos = np.arange(16)
ax.bar(x_pos - 0.2, mean_acts_0, 0.35, label='Digit 0', color='steelblue')
ax.bar(x_pos + 0.2, mean_acts_1, 0.35, label='Digit 1', color='coral')
ax.set_xlabel('Neuron')
ax.set_ylabel('Average activation')
ax.set_title('Which neurons fire for 0 vs 1?')
ax.legend()
ax.set_xticks(x_pos)
plt.tight_layout()
plt.show()

# Dead neurons?
all_acts = np.array(acts_0 + acts_1)
dead = (all_acts.max(axis=0) == 0)
print(f"\nDead neurons (never fire for any input): {dead.sum()} out of 16")
if dead.sum() > 0:
    print(f"Dead neuron indices: {np.where(dead)[0]}")

### Explore

Things you could try:
- Change the number of hidden neurons (4? 32? 64?). What happens to accuracy? To the weight images? To dead neurons?
- Try digits that look more similar: 3 vs 5, or 4 vs 9. Does the network struggle more?
- Pick a misclassified image. Plot its saliency map. Where is the network "looking" when it gets confused?
- Look at the weights of the output neuron. Which hidden neurons does it weight most heavily?

In [ ]:
# Your experiments here


### Challenge: all 10 digits

Can you extend this to classify all digits 0-9? You'll need:
- 10 output neurons instead of 1 (one per digit)
- A different loss: compare the 10 outputs to a "one-hot" target (e.g., digit 3 → `[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`)
- MSE still works: `loss = sum((output_k - target_k)^2 for k in range(10))`

Starter:

In [ ]:
# All digits
X_all = digits.data / 16.0
y_all = digits.target

# Network: 64 -> 16 -> 10
# random.seed(42)
# net10 = MLP(64, [16, 10])

# Training loop (SGD, one image at a time):
#   x = [Value(float(p)) for p in X_all[i]]
#   scores = net10(x)                       # list of 10 Values
#   target = [1.0 if k == y_all[i] else 0.0 for k in range(10)]
#   loss = sum((s - Value(t)) ** 2 for s, t in zip(scores, target))
#   ... zero_grad, backward, update ...
#
# This will take ~1-2 minutes to train. That's fine — you can see every gradient.
# For bigger images (28×28 = 784 pixels), micrograd would be too slow.
# That's what Part 3 is for.


---

## Part 3: Going Faster — Numpy Arrays (Optional)

Micrograd processes one number at a time. That's what makes it transparent — you can trace every multiplication, every gradient. But for 784-pixel images, it's too slow.

The fix: instead of wrapping every number in a `Value`, pack them into numpy arrays. The math is the same — same forward pass, same chain rule, same gradient descent. Just faster.

Micrograd is the algorithm. Arrays are the efficiency. *Everything else is just efficiency.*

### Fashion-MNIST

10 classes of clothing — t-shirts, trousers, sneakers, bags, etc. Each image is 28×28 = 784 pixels. 60,000 training images. Try doing this with micrograd.

In [ ]:
# Load Fashion-MNIST
# (torchvision downloads it; we just use it for the data, then work in numpy)
from torchvision import datasets
fmnist_train = datasets.FashionMNIST('data', train=True, download=True)
fmnist_test = datasets.FashionMNIST('data', train=False, download=True)

class_names = ['T-shirt', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

### Inspect the data

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    img = np.array(fmnist_train[i][0])
    label = fmnist_train[i][1]
    ax.imshow(img, cmap='gray')
    ax.set_title(class_names[label])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# What does one image look like as numbers?
img = np.array(fmnist_train[0][0])
print(f"Shape: {img.shape}")
print(f"Min pixel: {img.min()}, Max pixel: {img.max()}")
print("\nMiddle rows (zoom in — these are just numbers):")
print(img[10:18, 8:20])

In [ ]:
# Prepare data: flatten images, normalize pixels to 0-1
X_train = np.array([np.array(fmnist_train[i][0]).flatten() for i in range(len(fmnist_train))]) / 255.0
y_train = np.array([fmnist_train[i][1] for i in range(len(fmnist_train))])

X_test = np.array([np.array(fmnist_test[i][0]).flatten() for i in range(len(fmnist_test))]) / 255.0
y_test = np.array([fmnist_test[i][1] for i in range(len(fmnist_test))])

print(f"Training: {X_train.shape[0]} images, each {X_train.shape[1]} pixels")
print(f"Test: {X_test.shape[0]} images")

### Same algorithm, numpy arrays

Compare this to micrograd's forward/backward. The operations are identical — `@` is matrix multiply (many dot products at once), `np.maximum` is ReLU, the backward pass is the chain rule applied to each operation in reverse.

Every line of the backward pass is labeled with the derivative rule. No autograd, no `loss.backward()`. You can see exactly what's being computed.

In [ ]:
np.random.seed(42)
n_hidden = 64
n_classes = 10

W1 = np.random.randn(784, n_hidden) * 0.01
b1 = np.zeros(n_hidden)
W2 = np.random.randn(n_hidden, n_classes) * 0.01
b2 = np.zeros(n_classes)

In [ ]:
def forward(X):
    z1 = X @ W1 + b1            # linear: same as micrograd's sum(wi*xi) + b
    h = np.maximum(z1, 0)       # ReLU: same as micrograd's max(0, x)
    scores = h @ W2 + b2        # linear
    return z1, h, scores

def softmax(scores):
    exp_scores = np.exp(scores - scores.max(axis=1, keepdims=True))
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)

def cross_entropy_loss(probs, y):
    n = len(y)
    correct_probs = probs[range(n), y]
    return -np.log(correct_probs + 1e-8).mean()

In [ ]:
learning_rate = 0.1
batch_size = 128
epochs = 20
train_losses = []

for epoch in range(epochs):
    perm = np.random.permutation(len(X_train))
    epoch_loss = 0
    n_batches = 0
    
    for start in range(0, len(X_train), batch_size):
        idx = perm[start:start + batch_size]
        X_batch = X_train[idx]
        y_batch = y_train[idx]
        n = len(y_batch)
        
        # Forward
        z1, h, scores = forward(X_batch)
        probs = softmax(scores)
        loss = cross_entropy_loss(probs, y_batch)
        
        # Backward — same chain rule as micrograd, but with arrays
        dscores = probs.copy()
        dscores[range(n), y_batch] -= 1
        dscores /= n
        
        dW2 = h.T @ dscores              # d_loss/d_W2
        db2 = dscores.sum(axis=0)        # d_loss/d_b2
        dh = dscores @ W2.T              # chain rule into hidden
        dz1 = dh.copy()
        dz1[z1 <= 0] = 0                 # ReLU backward: same as micrograd
        dW1 = X_batch.T @ dz1            # d_loss/d_W1
        db1 = dz1.sum(axis=0)            # d_loss/d_b1
        
        # Update — gradient descent, same as micrograd's p.data -= lr * p.grad
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
        
        epoch_loss += loss
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    
    _, _, test_scores = forward(X_test)
    test_preds = test_scores.argmax(axis=1)
    accuracy = (test_preds == y_test).mean()
    
    print(f"Epoch {epoch+1:2d}: loss = {avg_loss:.4f}, test accuracy = {accuracy:.1%}")

In [ ]:
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training loss')
plt.show()

### Look inside: what did the neurons learn?

In [ ]:
# Visualize first-layer weights — same idea as the 8×8 weights above, but now 28×28
fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(W1[:, i].reshape(28, 28), cmap='RdBu', vmin=-W1.max(), vmax=W1.max())
    ax.axis('off')
plt.suptitle('First-layer weights (28×28) — what each neuron looks for', fontsize=14)
plt.tight_layout()
plt.show()

### Dead neurons

In [ ]:
z1_all, _, _ = forward(X_train)
activations = np.maximum(z1_all, 0)
dead = (activations.max(axis=0) == 0)
print(f"Dead neurons: {dead.sum()} out of {n_hidden}")
print(f"Alive neurons: {(~dead).sum()}")

### Misclassifications

In [ ]:
_, _, test_scores = forward(X_test)
test_preds = test_scores.argmax(axis=1)
wrong = np.where(test_preds != y_test)[0]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    idx = wrong[i]
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f"Pred: {class_names[test_preds[idx]]}\nTrue: {class_names[y_test[idx]]}", fontsize=9)
    ax.axis('off')
plt.suptitle('Misclassifications', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
confusion = np.zeros((10, 10), dtype=int)
for true, pred in zip(y_test, test_preds):
    confusion[true, pred] += 1

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(confusion, cmap='Blues')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(class_names, fontsize=9)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion matrix')
for i in range(10):
    for j in range(10):
        ax.text(j, i, confusion[i, j], ha='center', va='center',
                color='white' if confusion[i, j] > confusion.max()/2 else 'black', fontsize=8)
plt.tight_layout()
plt.show()

### Experiment (optional)

- Change `n_hidden` to 16 or 256. What happens to accuracy? Training speed? Dead neurons?
- Change `learning_rate` to 0.01 or 1.0. What breaks?
- Train for 50 epochs. Does accuracy keep improving, or plateau?
- Can you overfit? (Large network + many epochs. Compare train vs test accuracy.)

In [ ]:
# Your experiments here


---

## 🧩 Bonus Challenge

### The Entire GPT Algorithm

Someone wrote all of GPT — the complete algorithm behind ChatGPT — in [~200 lines of pure Python](https://gist.github.com/karpathy/8627fe009c40f57531cb18360106ce95). No PyTorch. No numpy. Just Python and math.

*"This file is the complete algorithm. Everything else is just efficiency."*

You won't understand all of it yet. The attention mechanism uses linear algebra you haven't taken. Skip those parts.

Your challenge: **how much of this can you read after what you learned this week?** Find the chain rule. Find the gradient descent. Find where it's doing the same thing as your polynomial regression — minimizing a loss by computing derivatives.

You'll be surprised how much of GPT is math you already know.

### Rabbit hole — if you want to understand backprop for real

[CS231n Lecture 4: Backpropagation](https://www.youtube.com/watch?v=i94OvYb6noo) (~75 min) — Andrej Karpathy teaching at Stanford. He explains backprop using computational graphs and the chain rule. Similar format to 3Blue1Brown, but at a Stanford undergrad level. You have Calc BC. You'll be fine.

**If you want to build it yourself after that:** Karpathy's [micrograd video](https://www.youtube.com/watch?v=VMj-3S1tku0) (2.5 hrs) — he implements the same `engine.py` you used in Part 1 from scratch, explaining every line. His pitch: *"If you know Python and have a vague recollection of taking some derivatives in high school, watch this and not understand backpropagation by the end, then I will eat a shoe."* You have Calc BC, not a vague recollection.

*(If you've seen 3Blue1Brown's neural net series — that's the visual intuition. The Stanford lecture is the math. The micrograd video is the code. Same idea, three angles.)*

### Advanced reading — how deep networks see

Your 2-layer network learns blurry templates. Deeper networks with **convolutional** layers (which share weights across spatial positions) learn something remarkable: the first layer detects **edges**, the next combines edges into **textures**, then into **object parts**, and so on.

These articles from [Distill.pub](https://distill.pub/) — the gold standard for interactive ML explanations — let you see through the eyes of a neural network:

- [**Feature Visualization**](https://distill.pub/2017/feature-visualization/) — what neurons at each layer "see." Edges → textures → patterns → objects. Interactive, visual, stunning.
- [**Curve Detectors**](https://distill.pub/2020/circuits/curve-detectors/) — a deep dive into individual neurons that detect curves and edges. Part of the Circuits thread investigating how networks build up visual understanding.
- [**Activation Atlases**](https://distill.pub/2019/activation-atlas/) — an explorable map of everything a network has learned. *"Compare the curved edge detectors of mixed4a with the bowls and cups of mixed5b."*